# Chapter 13 — Let the Program Reflect

**Book alignment:** DSPy From First Principles, Chapter 13

**Question this notebook isolates:** Do single-run accept/reject decisions fall inside the noise band where unchanged program state already moves?

In [ ]:
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported and constructed only; compile is never executed against a model

from common.data import canonical_split
from common.dspy_program import dspy_editorial_feedback_v1, to_dspy_example
from common.provider import canonical_dspy_lm_kwargs

## Rejection needs a noise model

GEPA proposed three children and rejected all three against one baseline measurement. Replay the comparison against the recorded run-to-run band of the *unchanged* baseline and ask which rejections survive it.

In [ ]:
NOISE_FLOOR = 0.015  # recorded run-to-run variation of the 11-case dev mean
baseline_measurements = (0.7893, 0.7992)  # two runs of unchanged state
session_baseline = 0.7992  # the session GEPA decided against
children = {1: 0.7792, 2: 0.7932, 3: 0.7899}

rows = []
for child, score in children.items():
    rows.append({
        "candidate": child,
        "score": score,
        "vs_session_baseline": round(score - session_baseline, 4),
        "vs_typical_baseline": round(score - baseline_measurements[0], 4),
        "inside_noise_of_typical": abs(score - baseline_measurements[0]) <= NOISE_FLOOR,
    })

rows

In [ ]:
assert session_baseline == max(baseline_measurements)  # decision used the top of the band
assert rows[1]["inside_noise_of_typical"] and rows[2]["inside_noise_of_typical"]
assert all(abs(r["vs_session_baseline"]) <= 0.020 for r in rows)

for r in rows:
    print(f"child {r['candidate']}: {r['score']:.4f} (vs session {r['vs_session_baseline']:+.4f}, "
          f"vs typical {r['vs_typical_baseline']:+.4f}, inside noise: {r['inside_noise_of_typical']})")
print("two of three rejected children are indistinguishable from the baseline")

## Feedback blindness survives verbosity

GEPA's advantage is language feedback — but the deterministic feedback path only names failures its branches test. Run the book's frozen feedback function on the ed-035 qualifier deletion and read what reflection would have received.

In [ ]:
split = canonical_split()
dev_by_id = {c.case_id: c for c in split.dev}
ed035 = dev_by_id["ed-035"]
bad = ed035.reference_rewrite.replace("regularly ", "")

# GEPA-construction check only: a lazy LM object (never called) plus the optimizer.
reflection_lm = dspy.LM(**{k: v for k, v in canonical_dspy_lm_kwargs().items() if k != "api_key"})
assert len(getattr(reflection_lm, "history", []) or []) == 0
optimizer = dspy.GEPA(
    metric=dspy_editorial_feedback_v1,
    max_metric_calls=1,
    reflection_lm=reflection_lm,
    track_stats=True,
)
assert hasattr(optimizer, "compile")

example = to_dspy_example(ed035)
result = dspy_editorial_feedback_v1(example, dspy.Prediction(rewritten_text=bad))
({"score": round(result.score, 4), "feedback": result.feedback})

In [ ]:
assert abs(result.score - 0.9667) < 1e-4  # same frozen scalar travels alongside
assert result.feedback == {"program": "no deterministic failure"}

print("feedback on the qualifier deletion:", result.feedback)
print("richer words around the same blind spot teach reflection nothing")

## What we earned

The governance loop works — propose, measure, keep the parent — but as evidence it is thinner than it looks: two rejected children sit inside the noise band of unchanged state, and the deterministic feedback certifies the ed-035 deletion as satisfactory. Accepting on noise and rejecting on noise are the same error in opposite directions, and none of these optimizers carries a noise model.

Notebook 14 / Chapter 14 holds everything fixed and changes exactly one thing — the objective — to test whether the diagnosis was right.